# 07 · Pitfalls and debugging

A catalogue of the ways research notebooks go quietly wrong. Each entry: what it is, code that
misbehaves, how to detect it, and the fix. Ends with a 10-minute audit checklist.

**What's in here**
- (a) Index alignment in arithmetic; `.values` breaking alignment
- (b) Assigning results from a filtered/sorted frame back to the original
- (c) Unsorted data before `shift` / `rolling` / `merge_asof`
- (d) Duplicated timestamps
- (e) SettingWithCopyWarning, copy vs view
- (f) Chained comparison on a Series
- (g) Object-dtype numbers
- (h) NaN semantics: `sum`, `mean`, `==`, groupby dropping NaN keys
- (i) Dtype drift: int → float, bool → object
- (j) `inplace=True` returns None
- (k) `df[mask]["x"] = 1` does nothing
- (l) Axis confusion
- (m) Merge duplicating rows
- (n) tz-naive vs tz-aware comparisons
- (o) Look-ahead leakage patterns
- (p) `to_datetime` day-first ambiguity
- (q) Float equality
- (r) Removed pandas 1.x APIs
- (s) Performance: `iterrows`, categoricals
- Audit checklist

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

import warnings

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
ts = df.set_index("time").sort_index()
print(ts.shape)

(17520, 5)


## (a) Arithmetic aligns on the index — mismatched indexes give NaN, not an error

`a - b` matches labels first. If `b` is a filtered subset, the non-overlapping rows become NaN
silently. Using `.values` (or `.to_numpy()`) turns alignment off entirely and matches by *position*,
which is worse: no NaNs, just wrong numbers.

**Detect:** count NaNs after every arithmetic step; check `a.index.equals(b.index)`.

In [2]:
a = ts["consumption_mwh"]
b = ts["consumption_mwh"].loc[ts.index.hour >= 6]          # subset: 18 of 24 hours

diff = a - b
print("aligned:  NaN count", diff.isna().sum(), "(the 6 missing hours per day)")

try:
    a.values - b.values                      # different lengths -> at least it fails loudly
except ValueError as e:
    print("with .values ->", type(e).__name__, ":", str(e)[:60])

# position-based match that *does* run and is silently wrong
c = a.iloc[1:].reset_index(drop=True)
d = a.iloc[:-1].reset_index(drop=True)
print("(c - d) is actually the hour-on-hour diff, not zero:", (c - d).abs().mean().round(1))
print("indexes identical?", a.index.equals(b.index))

aligned:  NaN count 4380 (the 6 missing hours per day)
with .values -> ValueError : operands could not be broadcast together with shapes (17520,
(c - d) is actually the hour-on-hour diff, not zero: 1240.7
indexes identical? False


## (b) Assigning a Series computed on a filtered / sorted frame back to the original

Assignment `df["x"] = s` aligns `s` on the index, so a sorted or filtered `s` lands in the right rows
and missing rows become NaN — good. But if `s` was built from `.values` or after `reset_index`, the
alignment is gone and values land in the wrong rows with no warning.

In [3]:
frame = ts[["consumption_mwh"]].copy()
sorted_by_cons = frame.sort_values("consumption_mwh")

# good: index alignment puts each rank back on its own timestamp
frame["rank_ok"] = sorted_by_cons["consumption_mwh"].rank()
# bad: .values assigns by position -> rank of the *sorted* order lands on chronological rows
frame["rank_bad"] = sorted_by_cons["consumption_mwh"].rank().values

print("rank_ok  vs actual order corr: %.3f" % frame["rank_ok"].corr(frame["consumption_mwh"]))
print("rank_bad vs actual order corr: %.3f  <- nonsense" % frame["rank_bad"].corr(frame["consumption_mwh"]))
frame.head(3)

rank_ok  vs actual order corr: 0.985
rank_bad vs actual order corr: -0.086  <- nonsense


,consumption_mwh,rank_ok,rank_bad
time,,,
2022-01-01 00:00:00+00:00,26858.4,4814.0,1.0
2022-01-01 01:00:00+00:00,26177.8,4107.0,2.0
2022-01-01 02:00:00+00:00,26229.4,4170.0,3.0


In [4]:
# subset assignment: NaN where the subset had no row (visible), vs .values raising (length mismatch)
subset = frame.loc[frame.index.hour == 12, "consumption_mwh"] * 2
frame["noon_x2"] = subset
print("noon_x2 non-NaN:", frame["noon_x2"].notna().sum(), "of", len(frame))
try:
    frame["noon_x2_bad"] = subset.values
except ValueError as e:
    print("with .values ->", str(e)[:70])

noon_x2 non-NaN: 730 of 17520
with .values -> Length of values (730) does not match length of index (17520)


## (c) Not sorting before `shift` / `rolling` / `merge_asof`

Row-based operations work on the *physical* order. Data that arrives shuffled (or sorted by another
column) produces lags that are not lags. `merge_asof` at least raises; `shift` and `rolling` do not.

**Detect:** `index.is_monotonic_increasing` right after loading and after every sort/merge/concat.

In [5]:
raw = pd.read_csv("../data/hourly_power_raw.csv")
raw["time"] = pd.to_datetime(raw["time"], utc=True)
print("raw sorted?", raw["time"].is_monotonic_increasing)

lag_unsorted = raw["consumption_mwh"].shift(1)
raw_sorted = raw.sort_values("time").drop_duplicates("time").reset_index(drop=True)
lag_sorted = raw_sorted["consumption_mwh"].shift(1)

print("corr(consumption, lag) unsorted: %.3f   sorted: %.3f" % (
    raw["consumption_mwh"].corr(lag_unsorted), raw_sorted["consumption_mwh"].corr(lag_sorted)))

raw sorted? False
corr(consumption, lag) unsorted: 0.004   sorted: 0.935


## (d) Duplicated timestamps

Duplicates make `shift(1)` return the same hour, double-count in `resample().sum()`, and make
`reindex` raise. Decide *why* they exist (re-published values? merge blow-up?) before choosing
`keep="last"` blindly.

In [6]:
dups = raw["time"].duplicated(keep=False)
print("duplicated timestamps:", dups.sum())
print(raw[dups].sort_values("time").head(4)[["time", "consumption_mwh"]])

dup_idx = raw.set_index("time").sort_index()
try:
    dup_idx.reindex(pd.date_range(dup_idx.index.min(), dup_idx.index.max(), freq="h"))
except Exception as e:
    print(type(e).__name__, "->", str(e)[:70])

fixed = dup_idx[~dup_idx.index.duplicated(keep="last")]
print("after dedup, unique index:", fixed.index.is_unique)

duplicated timestamps: 30
                           time  consumption_mwh
10516 2022-02-04 07:00:00+00:00          32724.5
10089 2022-02-04 07:00:00+00:00          32724.5
1106  2022-03-07 08:00:00+00:00          36737.5
2865  2022-03-07 08:00:00+00:00          36737.5
ValueError -> cannot reindex on an axis with duplicate labels
after dedup, unique index: True


## (e) SettingWithCopyWarning — copy or view?

`sub = df[df.x > 0]` may be a view or a copy; writing into `sub` may or may not reach `df`.
pandas warns. Fix: `.copy()` when you *want* an independent object, `.loc[mask, col] = value`
when you want to write into the original. (pandas 3 with Copy-on-Write removes the ambiguity: subsets
are always independent.)

In [7]:
work = ts[["consumption_mwh", "temp_c"]].copy()

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    sub = work[work.temp_c < 0]
    sub["cold"] = True                      # deliberately triggers the warning
    print("warnings raised:", [type(x.message).__name__ for x in w][:1])
print("'cold' reached the original?", "cold" in work.columns)

# the two intended versions
sub = work[work.temp_c < 0].copy(); sub["cold"] = True          # independent copy, no warning
work.loc[work.temp_c < 0, "cold"] = True                       # write into original
print("original now has 'cold':", work["cold"].sum(), "rows flagged")

warnings raised: ['SettingWithCopyWarning']
'cold' reached the original? False
original now has 'cold': 1139 rows flagged


## (f) `a < x < b` does not work on a Series

Python evaluates it as `(a < x) and (x < b)`, and `and` on a Series is ambiguous. Use `&` with
parentheses, or `.between()`.

In [8]:
t = ts["temp_c"]
try:
    mask = 0 < t < 10
except ValueError as e:
    print(type(e).__name__, "->", str(e)[:60])

mask = (t > 0) & (t < 10)
print("with &:", mask.sum(), "| between (inclusive):", t.between(0, 10).sum(),
      "| between exclusive:", t.between(0, 10, inclusive="neither").sum())

ValueError -> The truth value of a Series is ambiguous. Use a.empty, a.boo
with &: 7652 | between (inclusive): 7661 | between exclusive: 7652


## (g) Numbers stored as object dtype

One `"missing"` string in a CSV column turns the whole column into `object`. `mean()` then raises
(or, for `sum()`, concatenates strings). Use `pd.to_numeric(errors="coerce")` and count what became NaN.

**Detect:** `df.dtypes` immediately after `read_csv`; anything `object` that should be numeric is a red flag.

In [9]:
print(raw.dtypes["price_eur_mwh"])
try:
    raw["price_eur_mwh"].mean()
except TypeError as e:
    print("mean() ->", type(e).__name__, ":", str(e)[:60])

price = pd.to_numeric(raw["price_eur_mwh"], errors="coerce")
print("became NaN:", price.isna().sum(), "| offending values:", raw.loc[price.isna(), "price_eur_mwh"].unique())
print("mean now: %.2f" % price.mean())

object
mean() -> TypeError : Could not convert string '70.1623.19115.7980.5069.4946.8498.
became NaN: 100 | offending values: ['missing']
mean now: 98.52


## (h) NaN semantics

- `sum()` of all-NaN is **0**, not NaN (`min_count=1` to change that).
- `mean()` skips NaN — the denominator shrinks silently.
- `x == np.nan` is always False; use `isna()`.
- `groupby` drops NaN keys by default (`dropna=False` to keep them) — a whole category vanishes.

In [10]:
s = pd.Series([np.nan, np.nan])
print("sum of all-NaN:", s.sum(), "| with min_count=1:", s.sum(min_count=1))
print("mean skips NaN:", pd.Series([1, np.nan, 3]).mean(), "| count used:", pd.Series([1, np.nan, 3]).count())
print("np.nan == np.nan:", np.nan == np.nan, "| isna:", pd.isna(np.nan))

meters = pd.read_csv("../data/meters.csv")
print("\ntariff NaN rows:", meters.tariff.isna().sum())
print("default groupby sizes sum:", meters.groupby("tariff").size().sum(), "of", len(meters))
print(meters.groupby("tariff", dropna=False).size())

sum of all-NaN: 0.0 | with min_count=1: nan
mean skips NaN: 2.0 | count used: 2
np.nan == np.nan: False | isna: True

tariff NaN rows: 13
default groupby sizes sum: 287 of 300
tariff
Fixed       143
TOU          50
Variable     94
NaN          13
dtype: int64


## (i) Dtype drift: int → float, bool → object

Introducing a single NaN into an int column upcasts it to float (ids like `100000` become `100000.0`,
and may print as `1e5`). NaN in a bool column gives `object`, which breaks `~mask`. Use nullable
dtypes (`"Int64"`, `"boolean"`) if you need NaN with ints/bools.

In [11]:
ids = pd.Series([100000, 100001, 100002])
ids_nan = ids.copy().astype(float); ids_nan[1] = np.nan
print("int:", ids.dtype, "| after NaN:", ids_nan.dtype, ids_nan.tolist())
print("nullable:", pd.Series([100000, None, 100002], dtype="Int64").tolist())

flags = pd.Series([True, False, None])
print("bool with None ->", flags.dtype)
try:
    print(~flags)
except TypeError as e:
    print("~ on object ->", str(e)[:50])
print("nullable boolean:", (~pd.Series([True, False, None], dtype="boolean")).tolist())

int: int64 | after NaN: float64 [100000.0, nan, 100002.0]
nullable: [100000, <NA>, 100002]
bool with None -> object
~ on object -> bad operand type for unary ~: 'NoneType'
nullable boolean: [False, True, <NA>]


## (j) `inplace=True` returns None

`df = df.dropna(inplace=True)` sets `df` to `None`. Either assign the result or use inplace, never both.
Prefer assignment: it chains and works with Copy-on-Write.

In [12]:
tmp = ts[["temp_c"]].copy()
result = tmp.dropna(inplace=True)
print("inplace returns:", result)
tmp = ts[["temp_c"]].copy().dropna()          # preferred
print(type(tmp).__name__, tmp.shape)

inplace returns: None
DataFrame (17520, 1)


## (k) `df[mask]["x"] = 1` does nothing

Chained indexing writes into a temporary. Use `df.loc[mask, "x"] = 1`.

In [13]:
work = ts[["price_eur_mwh"]].copy()
work["spike"] = 0
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    work[work.price_eur_mwh > 200]["spike"] = 1        # deliberately wrong
print("flagged via chained indexing:", work["spike"].sum())
work.loc[work.price_eur_mwh > 200, "spike"] = 1
print("flagged via .loc:", work["spike"].sum())

flagged via chained indexing: 0
flagged via .loc: 71


## (l) Axis confusion

`axis=0` = down the rows (per column); `axis=1` = across the columns (per row). `drop("col")`
defaults to rows, so `drop("temp_c")` raises KeyError; use `columns=`. `apply(axis=1)` is row-wise
(and slow). `concat(axis=1)` is side by side.

In [14]:
small = ts[["consumption_mwh", "temp_c"]].head(3)
print("mean(axis=0):", small.mean(axis=0).round(1).tolist(), " <- per column")
print("mean(axis=1):", small.mean(axis=1).round(1).tolist(), " <- per row (meaningless here)")
try:
    small.drop("temp_c")
except KeyError as e:
    print("drop('temp_c') ->", "KeyError: not in the row index")
print(small.drop(columns="temp_c").columns.tolist())

mean(axis=0): [26421.9, -0.4]  <- per column
mean(axis=1): [13429.3, 13088.8, 13114.1]  <- per row (meaningless here)
drop('temp_c') -> KeyError: not in the row index
['consumption_mwh']


## (m) Merge duplicating rows

Covered in depth in notebook 05. The one-line defence: `validate="many_to_one"` and an assert on the
row count.

In [15]:
readings = pd.read_csv("../data/meter_readings_daily.csv")
dim = pd.concat([meters, meters.iloc[[5]]])                       # a duplicated dimension row
merged = readings.merge(dim, on="meter_id", how="left")
print("rows before", len(readings), "after", len(merged))
try:
    assert len(merged) == len(readings), "left merge changed the row count"
except AssertionError as e:
    print("AssertionError:", e)

rows before 107503 after 107864
AssertionError: left merge changed the row count


## (n) tz-naive vs tz-aware comparisons

Comparing or slicing an aware index with a naive Timestamp raises (`.loc` with strings is fine, it
interprets them in the index's zone). Mixing both in one project is the root cause; pick UTC everywhere.

In [16]:
cut = pd.Timestamp("2023-01-01")
try:
    ts.loc[ts.index >= cut]
except TypeError as e:
    print(type(e).__name__, "->", str(e)[:70])

print("string slice works:", len(ts.loc["2023-01-01":]))
print("aware timestamp works:", len(ts.loc[ts.index >= pd.Timestamp("2023-01-01", tz="UTC")]))

TypeError -> Invalid comparison between dtype=datetime64[ns, UTC] and Timestamp
string slice works: 8760
aware timestamp works: 8760


## (o) Look-ahead leakage patterns

Five ways future information enters a feature matrix, each of which makes an out-of-sample metric look
better than it will be in production:

1. `rolling(24).mean()` without `shift(1)` — includes the current target.
2. `target = y.shift(-h)` then `X.dropna()` and `y.dropna()` **separately** — rows no longer aligned.
3. Standardising with full-sample mean/std before the train/test split.
4. `fillna(method="bfill")` / `interpolate()` — fills gaps with *later* values.
5. `resample(label="right")` — bar timestamp is the *end* of the interval it summarises.

**Interview check:** *"Your R² went from 0.71 to 0.93 after adding this feature. Why might that be bad news?"*

In [17]:
y = ts["consumption_mwh"]
X = pd.DataFrame({"lag1": y.shift(1), "temp": ts["temp_c"]})

# 1. rolling without shift: the feature contains the target; the shorter the window, the bigger the flattery
for w in [2, 3, 6, 24]:
    print("window %2d   corr rolling(w): %.3f   corr shift(1).rolling(w): %.3f" % (
        w, y.corr(y.rolling(w).mean()), y.corr(y.shift(1).rolling(w).mean())))

window  2   corr rolling(w): 0.984   corr shift(1).rolling(w): 0.873
window  3   corr rolling(w): 0.943   corr shift(1).rolling(w): 0.798
window  6   corr rolling(w): 0.749   corr shift(1).rolling(w): 0.573
window 24   corr rolling(w): 0.495   corr shift(1).rolling(w): 0.488


In [18]:
# 2. separate dropna on X and y -> misaligned rows if you then use .values / fit on arrays
target = y.shift(-6)                       # 6-hour-ahead target
Xd, yd = X.dropna(), target.dropna()
print("len X:", len(Xd), "len y:", len(yd), "| indexes equal:", Xd.index.equals(yd.index))

# fix: align once, on a single frame
data = X.assign(target=target).dropna()
print("aligned rows:", len(data))

len X: 17519 len y: 17514 | indexes equal: False
aligned rows: 17513


In [19]:
# 3. full-sample standardisation leaks test-period mean/std into training
split = int(len(data) * 0.8)
train, test = data.iloc[:split], data.iloc[split:]
leaky = (data["temp"] - data["temp"].mean()) / data["temp"].std()           # uses test period
clean = (data["temp"] - train["temp"].mean()) / train["temp"].std()          # train stats only
print("train-period mean of standardised temp: leaky %.4f  clean %.4f  (clean is exactly 0 by construction)" % (
    leaky.iloc[:split].mean(), clean.iloc[:split].mean()))
print("the leaky version's training features already 'know' the test period's mean and spread;")
print("the effect is small here but the principle scales: fit every transformer on train only")

train-period mean of standardised temp: leaky -0.0064  clean -0.0000  (clean is exactly 0 by construction)
the leaky version's training features already 'know' the test period's mean and spread;
the effect is small here but the principle scales: fit every transformer on train only


In [20]:
# 4. bfill / interpolate use the future; ffill does not
g = pd.Series([1.0, np.nan, np.nan, 4.0], index=pd.date_range("2022-01-01", periods=4, freq="h"))
pd.DataFrame({"raw": g, "ffill (ok)": g.ffill(), "bfill (future)": g.bfill(), "interpolate (future)": g.interpolate()})

,raw,ffill (ok),bfill (future),interpolate (future)
2022-01-01 00:00:00,1.0,1.0,1.0,1.0
2022-01-01 01:00:00,NaN,1.0,4.0,2.0
2022-01-01 02:00:00,NaN,1.0,4.0,3.0
2022-01-01 03:00:00,4.0,4.0,4.0,4.0


In [21]:
# 5. label="right": the 4h bar stamped 04:00 contains 00:00-03:00 data
s = y.loc["2022-06-01 00:00":"2022-06-01 07:00"]
pd.DataFrame({"label_left": s.resample("4h").mean(), "label_right": s.resample("4h", label="right").mean()}).round(0)

,label_left,label_right
time,,
2022-06-01 00:00:00+00:00,22115.0,NaN
2022-06-01 04:00:00+00:00,24798.0,22115.0
2022-06-01 08:00:00+00:00,NaN,24798.0


## (p) `to_datetime` and day-first ambiguity

`"03/04/2022"` is 3 April in the UK and 4 March in the US. pandas guesses month-first unless told. Pass
`format=` (fastest, unambiguous) or `dayfirst=True`. Mixed formats in one column → parse with
`errors="coerce"` and inspect the NaTs.

In [22]:
s = pd.Series(["03/04/2022", "05/04/2022"])
print("default (month first):", pd.to_datetime(s).dt.strftime("%d %b").tolist())
print("dayfirst=True        :", pd.to_datetime(s, dayfirst=True).dt.strftime("%d %b").tolist())
print("format='%d/%m/%Y'    :", pd.to_datetime(s, format="%d/%m/%Y").dt.strftime("%d %b").tolist())

# pandas infers the format from the first value; a later "13/04" that contradicts it raises
s2 = pd.Series(["03/04/2022", "13/04/2022"])
try:
    pd.to_datetime(s2)
except Exception as e:
    print(type(e).__name__, "->", str(e)[:80])

mixed = pd.Series(["2022-01-01 00:00", "01/02/2022", "not a date"])
parsed = pd.to_datetime(mixed, errors="coerce", format="mixed", dayfirst=True)
print("NaT rows:", mixed[parsed.isna()].tolist())

default (month first): ['04 Mar', '04 May']
dayfirst=True        : ['03 Apr', '05 Apr']
format='%d/%m/%Y'    : ['03 Apr', '05 Apr']
ValueError -> time data "13/04/2022" doesn't match format "%m/%d/%Y", at position 1. You might
NaT rows: ['not a date']


## (q) Float equality

`0.1 + 0.2 != 0.3`. Never filter with `==` on computed floats; use `np.isclose` or round first.
Also affects merges on float keys and `drop_duplicates` on float columns.

In [23]:
x = pd.Series([0.1 + 0.2, 0.3])
print("== 0.3:", (x == 0.3).tolist(), "| isclose:", np.isclose(x, 0.3).tolist())
print("round(10) then ==:", (x.round(10) == 0.3).tolist())

== 0.3: [False, True] | isclose: [True, True]
round(10) then ==: [True, True]


## (r) Removed in pandas 2.x (mention only)

- `df.append(...)` → `pd.concat([df, other])`
- `Series.iteritems()` → `.items()`
- `df.lookup` → `df.to_numpy()[rows, cols]` or `melt`
- `Series.mad()` → `(s - s.mean()).abs().mean()`
- freq alias `"H"` → `"h"`, `"T"` → `"min"`, `"S"` → `"s"` (deprecated, still works with a warning)

In [24]:
print("pandas", pd.__version__)
print(hasattr(pd.DataFrame, "append"), hasattr(pd.Series, "iteritems"))

pandas 2.3.3
False False


## (s) Performance: `iterrows` vs vectorised; categoricals for memory

`iterrows` is 100–1000× slower than vectorised arithmetic and returns each row as a Series with a
possibly upcast dtype. String columns with few distinct values shrink dramatically as `category`.

In [25]:
import time
sample = ts.head(5000).copy()

t0 = time.perf_counter()
hdd = [max(0.0, 15 - r["temp_c"]) for _, r in sample.iterrows()]
t_iter = time.perf_counter() - t0

t0 = time.perf_counter()
hdd_vec = np.clip(15 - sample["temp_c"], 0, None)
t_vec = time.perf_counter() - t0
print("iterrows %.3fs   vectorised %.5fs   speedup x%.0f" % (t_iter, t_vec, t_iter / t_vec))

reg = pd.Series(np.random.default_rng(0).choice(["London", "North", "Wales"], 200_000))
print("object %.1f MB  ->  category %.1f MB" % (reg.memory_usage(deep=True) / 1e6,
                                                 reg.astype("category").memory_usage(deep=True) / 1e6))

iterrows 0.144s   vectorised 0.00121s   speedup x120


object 12.5 MB  ->  category 0.2 MB


## 10-minute notebook audit checklist

Read this top to bottom when handed someone else's notebook.

**Loading**
1. `df.dtypes` — any `object` column that should be numeric or datetime?
2. `df.shape`, `df.isna().sum()`, `df.duplicated().sum()` — printed, not assumed.
3. Timestamps: parsed? tz-aware? which zone? `is_monotonic_increasing`, `is_unique`, `infer_freq`.

**Structure**
4. What is one row? Is the sampling interval constant? Where are the gaps?
5. Sentinels (`-999`, `0` for missing, `"missing"`) converted to NaN?
6. Every merge: key dtypes, `validate=`, row count before vs after, `indicator=`.

**Features**
7. Every `rolling` / `expanding` preceded by `shift(1)` (or justified).
8. Every `shift(-h)` target: aligned with X in the *same* frame before `dropna`.
9. Any `bfill` / `interpolate` / `label="right"` / centred windows? → look-ahead.
10. Scaler / imputer fitted on train only.
11. Does each variable exist at prediction time? (forecasts vs actuals, publication lags)

**Split & evaluation**
12. Chronological split (no shuffle); gap between train and test ≥ horizon for overlapping labels.
13. Metric compared with a naive baseline (`lag24`, seasonal mean).
14. Suspiciously good result → look for leakage before celebrating.
15. Residuals grouped by hour / dow / month / regime — where is the model systematically wrong?
16. Coefficients: signs and magnitudes plausible? One period driving the fit?

**Hygiene**
17. `inplace=True` with assignment? chained-indexing writes? `.values` on aligned data?
18. Cells run top to bottom in a fresh kernel and give the same numbers.